# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kailaswadje/FlyRank-Internship_ML_Assignment_01_Week_01/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Provisional lane: Lane 4 — CTR / Engagement Opportunity Scoring.**

I'm picking this over the other three because the starter data shows real, sizeable spread *within the same position tier* — see Section 3. Two pages both sitting on page 1 of search results can have very different click-through rates, and that gap is exactly what this lane is built to rank. It also lines up with the kind of work I want to get good at this internship: not just detecting that something changed (Lane 1), not clustering pages into types (Lane 3), but building a ranked, evidence-backed list that a real reviewer could act on with limited time — which is closer to the retrieval/ranking and evaluation work I've been doing on my dissertation (comparing candidates against a baseline, validating with proper held-out splits, not just reporting a headline number).

I'm treating this as provisional, not final — the guide gives me until end of Week 4 to confirm or switch, and I want to see the CTR-vs-engagement split more closely and check the warehouse-scale numbers before I commit fully.

In [10]:
import pandas as pd

pd.set_option('display.width', 120)

df = pd.read_csv('/content/content_refresh_anonymized.csv')
print('starter dataset shape:', df.shape)

starter dataset shape: (30000, 44)


## 2. The question: decision, action, cost of a wrong call

**Question:** Among pages that are already visible in search (indexed, earning impressions), which ones are under-capturing clicks or engagement *relative to what their position tier and volume would predict* — and should therefore be reviewed first?

**Decision this improves:** which handful of pages a content reviewer opens first this week, out of a much larger pool of "visible" pages, when they only have time to review a limited number.

**Who acts, and what they do:** a FlyRank content/SEO reviewer. They take the top of the ranked queue, open each page, and — depending on the reason code attached — either rewrite the title/meta description (CTR gap while position is strong) or review on-page content and layout (engagement/scroll gap despite decent traffic).

**Cost of a wrong recommendation:**
- **False positive** (page flagged as underperforming, but it isn't really): wastes a reviewer's limited hour on a page that didn't need it — and that hour is now not spent on a page that genuinely did.
- **False negative** (a genuinely underperforming, high-impression page is missed): the page keeps quietly leaking clicks or engagement every day it isn't reviewed, and nobody ever finds out, because nothing in the raw data "alerts" on its own — someone has to go look.

Because reviewer time is the scarce resource, this is fundamentally a ranking-quality problem, not an accuracy problem — precision at the top of the list (precision@K) matters far more than getting every borderline row right.

In [11]:
# No additional numbers needed for this section — the framing above is answered in words,
# per the framing-ml-problems skill ("decision, action, cost" are qualitative until Section 3
# grounds the lane choice in real data).

## 3. Quick look at the data (2-3 real numbers)

Loaded straight from `data/raw/content_refresh_anonymized.csv` (30,000 rows × 44 columns — matches the lane guide). No numbers below are invented; every one is computed in the cells beneath this markdown.

In [12]:
# Apply the same filters the starter pipeline uses (impressions_90d > 0, content_age_days >= 90),
# dedup by content_id
filtered = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].drop_duplicates('content_id')
print('rows after starter filters:', len(filtered))
print('unique clients represented:', filtered['client_id'].nunique())

# avg_position == 0 means "no data", not rank zero -- confirms I've read the data-dictionary gotcha
no_position_data = (df['avg_position'] == 0).sum()
print('rows with avg_position == 0 (no data, per data dictionary):', no_position_data)

# "Visible" pool for this lane: ranked 1-20 with meaningful impression volume
visible = filtered[(filtered['avg_position'] > 0) & (filtered['avg_position'] <= 20) & (filtered['impressions_90d'] >= 500)]
print('candidate pool (position 1-20, impressions_90d >= 500):', len(visible))

rows after starter filters: 30000
unique clients represented: 32
rows with avg_position == 0 (no data, per data dictionary): 1205
candidate pool (position 1-20, impressions_90d >= 500): 12023


In [13]:
# The core justification for this lane: does position tier alone explain CTR?
# If it did, every tier's CTR spread would be tight. Checking ALL 5 tiers, not just one.
visible_pos = filtered[filtered['avg_position'] > 0]
tier_table = visible_pos.groupby('position_tier')['ctr'].agg(['count', 'mean', 'median', 'std']).round(3)
print('CTR by position tier (all 5 tiers):')
print(tier_table)
print()
print('-> every tier has a std larger than its mean -- position tier alone does not pin down CTR.')
print('   That gap, repeated across every tier and not just one, is the real evidence for this lane.')

CTR by position tier (all 5 tiers):
               count   mean  median     std
position_tier                              
deep            1319  0.150    0.00   1.613
page_1         11814  0.652    0.16   3.089
page_3_5        7242  0.222    0.03   2.148
striking        7304  0.323    0.11   1.387
top_3           1116  2.764    0.00  10.812

-> every tier has a std larger than its mean -- position tier alone does not pin down CTR.
   That gap, repeated across every tier and not just one, is the real evidence for this lane.


In [14]:
# Zooming into page_1 specifically, since it's the tier where you'd most expect CTR to be
# uniformly high (top-of-page real estate) -- and it still isn't.
page1 = filtered[filtered['position_tier'] == 'page_1']
low_ctr_on_page1 = (page1['ctr'] < 0.2).sum()
strong_ctr_on_page1 = (page1['ctr'] > 1.0).sum()
print(f"page_1 tier (n={len(page1)}): {low_ctr_on_page1} pages have ctr < 0.2, "
      f"while {strong_ctr_on_page1} have ctr > 1.0 -- same tier, very different outcomes")
print()

# Using the starter's own low_ctr_visible_page reason code definition
low_ctr_candidates = filtered[(filtered['impressions_90d'] >= 500) & (filtered['avg_position'] > 0)
                               & (filtered['avg_position'] <= 20) & (filtered['ctr'] < 0.5)]
pct_ctr = 100 * len(low_ctr_candidates) / len(filtered)
print(f"low_ctr_visible_page candidates (starter reason-code definition): {len(low_ctr_candidates)} "
      f"({pct_ctr:.1f}% of the filtered dataset)")

# Using the starter's own low_engagement_visible_page reason code definition:
# sessions_90d >= 30 and (engagement_rate < 30 or scroll_rate < 30)
low_eng_candidates = filtered[(filtered['sessions_90d'] >= 30)
                               & ((filtered['engagement_rate'] < 30) | (filtered['scroll_rate'] < 30))]
pct_eng = 100 * len(low_eng_candidates) / len(filtered)
print(f"low_engagement_visible_page candidates (starter reason-code definition): {len(low_eng_candidates)} "
      f"({pct_eng:.1f}% of the filtered dataset)")

# How much do these two candidate pools overlap? If mostly separate, that's evidence CTR
# and engagement are two genuinely different problems, not one signal wearing two names.
overlap = len(set(low_ctr_candidates['content_id']) & set(low_eng_candidates['content_id']))
combined = pd.concat([low_ctr_candidates, low_eng_candidates]).drop_duplicates('content_id')
print()
print(f"overlap between the two reason codes: {overlap} pages "
      f"({100*overlap/len(combined):.1f}% of their combined pool)")
print(f"combined CTR-or-engagement candidate pool: {len(combined)} ({100*len(combined)/len(filtered):.1f}% of filtered data)")
print("-> mostly non-overlapping: CTR problems and engagement problems are largely different pages,")
print("   which argues for scoring them as two related but distinct signals, not one blended score.")

page_1 tier (n=11814): 6520 pages have ctr < 0.2, while 1005 have ctr > 1.0 -- same tier, very different outcomes

low_ctr_visible_page candidates (starter reason-code definition): 9759 (32.5% of the filtered dataset)
low_engagement_visible_page candidates (starter reason-code definition): 7113 (23.7% of the filtered dataset)

overlap between the two reason codes: 3418 pages (25.4% of their combined pool)
combined CTR-or-engagement candidate pool: 13454 (44.8% of filtered data)
-> mostly non-overlapping: CTR problems and engagement problems are largely different pages,
   which argues for scoring them as two related but distinct signals, not one blended score.


## 4. Careful words: what I can and can't claim

**What this work will be able to say:** observed, position-tier-adjusted patterns in click and engagement behavior on this anonymized starter slice (and, once validated, the warehouse release); a ranked, reason-coded list of review candidates suitable for decision support, where "high in the ranking" means "worth a human look first," not "guaranteed to be a problem."

**What it will never say:** that a title or meta rewrite *caused* a CTR increase (that needs a genuine experiment, not this data); that any result reveals a Google ranking factor; that AI referral behavior means anything about "AI visibility" beyond sessions someone actually clicked into; that a page flagged here is a guaranteed win if fixed. All numbers above come from a single anonymized 30,000-row snapshot (32 clients) — they motivate the lane, they don't yet validate a model. The 32.5% flagged by a flat CTR threshold, and the largely separate 23.7% flagged by a flat engagement threshold, is itself evidence that a naive single-cutoff rule is too blunt for a limited-capacity review queue — which is precisely why this needs a ranked, tier-adjusted approach that treats CTR and engagement as related but distinct signals, rather than one combined score.

In [15]:
# No additional numbers needed for this section -- the caveats above are qualitative,
# and are grounded in the flat-threshold results already computed in Section 3
# (32.5% CTR-flagged, 23.7% engagement-flagged, mostly non-overlapping) -- evidence
# that a smarter, tier-aware ranking is needed rather than a single cutoff.